# D1-04 First LCA from demand to score in Brightway

This notebook is the main Day 1 milestone: we bootstrap the course project from a prepared Brightway archive, import the bundled BAFU workbook, inspect the resulting database, and run a first LCA.

The hands-on exercises use BAFU throughout the course. We use `ecoinvent-3.10-biosphere` only as a convenient starting project because it already contains `biosphere3` and a populated set of LCIA methods.


## Learning goals

- Install or switch to the course `brightway` project from a prepared archive.
- Import the bundled BAFU workbook with `bw2io.ExcelImporter`.
- Understand the role of strategies, biosphere matching, statistics, and database writing.
- Inspect the LCIA methods that are already available in the project.
- Inspect the imported database by searching for activities and exchanges.
- Run a first LCA score with an explicitly chosen activity and method.
- Recognize how the Day 2 `ecoinvent 3.12 cutoff` import fits into the same overall workflow.


## Background references

- Mutel, C. (2017). *Brightway: An open source framework for life cycle assessment*. Journal of Open Source Software, 2(12), 236. https://doi.org/10.21105/joss.00236
- Database of the Swiss Federal Administration, FOEN:20XY, Federal Office for the Environment, 2025.


## 1) Install or switch to the course project

We start from a prepared Brightway project archive instead of building `biosphere3` and the LCIA methods from scratch.

The call `bi.remote.install_project('ecoinvent-3.10-biosphere', project_name)` downloads the archive on first use, restores it as a new project, and gives us a project that already contains `biosphere3` and many LCIA methods.
If the project already exists locally, we simply switch to it.


In [ ]:
from pathlib import Path
from pprint import pprint

import numpy as np
import pandas as pd
import bw2calc as bc
import bw2data as bd
import bw2io as bi

# this is just to let dataframe print fully
pd.set_option('display.max_colwidth', None)

In [ ]:
# let's choose a project name
project_name = "paris-lca-course-2026"

# and make a list of existing projects
existing_projects = [project.name for project in bd.projects]

In [ ]:
# if the project is already part of the projects' list, then our project already exists
# in this case, we only need to "switch" to it
if project_name in existing_projects:
    bd.projects.set_current(project_name)
    print('Switched to existing project:', bd.projects.current)

# if not, then we create it
else:
    bi.remote.install_project(project_key, project_name)
    # and we switch to it
    bd.projects.set_current(project_name)
    print('Installed and switched to project:', bd.projects.current)

In [ ]:
# this is needed later when importing LCI databases
bi.create_core_migrations()

print('Databases available now:', list(bd.databases))
print('LCIA methods available now:', len(bd.methods))

Let's check the size of `ecoinvent-3.10-biosphere`

In [ ]:
len(bd.Database("ecoinvent-3.10-biosphere"))

The prepared project already contains the core biosphere database and a large set of LCIA methods.
So the hands-on import work in this notebook is focused on the BAFU technosphere database.


## 2) Confirm the bundled BAFU workbook

The BAFU database we will be using is available as a workbook, contained in this repository:

- `data/lci-bafu.xlsx`


In [ ]:
bafu_file = Path('data/lci-bafu.xlsx')
if not bafu_file.exists():
    bafu_file = Path('../../data/lci-bafu.xlsx')

print('Workbook path:', bafu_file.resolve())
print('Found:', bafu_file.exists())
if bafu_file.exists():
    print('Workbook size [MB]:', round(bafu_file.stat().st_size / 1024 / 1024, 2))


## 3) Import pipeline: `ExcelImporter` -> strategies -> biosphere matching -> statistics -> database writing

This is the import sequence to understand in this notebook:

1. Create an `ExcelImporter` object from the workbook path.
2. Apply the standard import strategies (data cleanup, mostly).
3. Match biosphere flows against `biosphere3`.
4. Check import statistics and unlinked exchanges.
5. Write the database to the current project.


In [ ]:
database_name = 'bafu'

# create the ExcelImporter object
importer = bi.ExcelImporter(str(bafu_file))
print('Importer type:', type(importer).__name__)
print('Raw datasets loaded:', len(importer.data))


In [ ]:
# data cleanup
importer.apply_strategies()
print('Applied import strategies.')

In [ ]:
importer.match_database(fields=('name', 'reference product', 'location'))

In [ ]:
# now, we try to match the biosphere exchanges with the flows of the `biosphere` database
# on the basis of 'name', 'unit', 'categories'
importer.match_database('ecoinvent-3.10-biosphere', fields=('name', 'unit', 'categories'))
print('Matched biosphere exchanges against biosphere3.')

In [ ]:
# we check the status of exchanges linking
importer.statistics()

`0 unique unlinked edges (0 total)` is usually the sweet message you want to see.  
This means we can go ahead and write the database to the project.

In [ ]:
del bd.databases["bafu"]

In [ ]:
importer.write_database()

## 4) Inspect the LCIA methods already available in the project

Because the project archive came from `ecoinvent-3.10-biosphere`, the LCIA methods are already in place.
So, unlike the previous version of this notebook, we do not import LCIA methods from local Excel files here.


In [ ]:
preferred_method = ('EF v3.1', 'climate change', 'global warming potential (GWP100)')

print('Databases available:', list(bd.databases))
print('Number of LCIA methods available:', len(bd.methods))
print('Preferred climate-change method available:', preferred_method in bd.methods)


Let's quickly inspect a few available methods before choosing one for the first LCA.


In [ ]:
sample_methods = sorted(bd.methods)[:10]
climate_methods = [m for m in sorted(bd.methods) if 'climate change' in ' | '.join(m).lower()]

print('First 10 LCIA methods in the project:')
for method in sample_methods:
    print('-', ' | '.join(method))

print('\nFirst climate-related methods:')
for method in climate_methods[:10]:
    print('-', ' | '.join(method))


### REMINDER: how to inspect an LCIA method?

In [ ]:
my_method = bd.methods.random()
my_method

In [ ]:
method_obj = bd.Method(my_method)
method_data = method_obj.load()

In [ ]:
print('Method metadata:')
pprint(method_obj.metadata)

In [ ]:
# an empty list, to store the results
lcia_data = []

for flow_id, cf in method_data:
    flow = bd.get_activity(flow_id)
    lcia_data.append(
        [
            flow["name"],
            cf
        ]
    )

pd.DataFrame(lcia_data, columns=["flow name", "cf"])

## 5) Import your own LCIA method

We will use bw2io.LCIAExcelImporter to import a user-defined LCIA method file.  
You can find those files under `data/LCIA methods`. We will import `01__acidification__accumulated_exceedance_AE.xlsx`.

In [ ]:
lcia_method_file = Path('data/LCIA methods/01__acidification__accumulated_exceedance_AE.xlsx')
if not lcia_method_file.exists():
    lcia_method_file = Path('../../data/LCIA methods/01__acidification__accumulated_exceedance_AE.xlsx')

importer = bi.ExcelLCIAImporter(
    str(lcia_method_file),
    name=("EF 3.0", "Acidification"),
    description="Some method about ocean acidification",
    unit="kg SO2-eq."
)


In [ ]:
# data cleanup
importer.apply_strategies()

In [ ]:
# let's check if we have any unlinked CFs
importer.statistics()

In [ ]:
# we're good to go!
importer.write_methods()

In [ ]:
# let's confirm that our method is available
("EF 3.0", "Acidification") in bd.methods

In [ ]:
# let's inspect it
acid_method = bd.Method(("EF 3.0", "Acidification"))
pprint(acid_method.metadata)

In [ ]:
for flow_id, cf in acid_method.load():
    flow = bd.get_activity(flow_id)
    print(flow["name"], cf)

## 6) Inspect the imported database

Now we reconnect the import workflow from the previous notebooks: project -> database -> activity -> exchange.

A good way to start is to search the database by name, then refine with additional filters such as `location`.

In [ ]:


starter_queries = [
    'operation, passenger car, petrol',
    'passenger car, gasoline',
    'gasoline',
]
starter_hits = []
used_query = None

for query in starter_queries:
    hits = db.search(query, limit=10)
    if hits:
        starter_hits = hits
        used_query = query
        break

print('Search query used:', used_query)

pd.DataFrame(
    [
        {
            'name': act['name'],
            'location': act.get('location'),
            'unit': act.get('unit'),
        }
        for act in starter_hits
    ]
)

In [ ]:
db = bd.Database("bafu")
print('Number of activities in bafu:', len(db))

We can use `bw2data.search()` to look for activities (this, naturally, works also to search the `biosphere` database).  
You can always display the arguments and teh docstrings of a function like so:

In [ ]:
db.search?
# db.search() is quite complex if it needs to be...

In [ ]:
db.search(
    "gasoline",
    filter={
        "location": "RER",
        "reference product": "car"
    },
    limit=10
)

A `search` gets you close quickly. To be more explicit, you can iterate through the database yourself and combine several criteria in a list comprehension.

In [ ]:
gasoline_car_candidates = [
    act
    for act in db
    if 'gasoline' in act['name'].lower()
    and any(term in act['reference product'].lower() for term in ['car', 'gasoline'])
    and act["location"] == "RER"
    and act["unit"] == "kilometer"
]

pd.DataFrame(
    [
        {
            'name': act['name'],
            'location': act.get('location'),
            'unit': act.get('unit'),
        }
        for act in gasoline_car_candidates[:10]
    ]
)

Geography filters are often more reliable than free-text search alone. For example, you can ask for passenger-car activities in Switzerland (`CH`), Europe (`RER`), or global datasets (`GLO`).

In [ ]:
car_by_location = [
    act
    for act in db
    if 'passenger car' in act['name'].lower()
    and act.get('location') in {'CH', 'RER'}
    and act["unit"] == "kilometer"
]

pd.DataFrame(
    [
        {
            'name': act['name'],
            'location': act.get('location'),
            'unit': act.get('unit'),
        }
        for act in car_by_location[:15]
    ]
)

Let's choose one activity relating to driving a gasoline car and inspect its metadata and exchanges. Here we look at an `operation` activity, because it gives a nice mix of technosphere input(s) and biosphere emissions.

In [ ]:
selected_activity = gasoline_car_candidates[0]

pprint(selected_activity.as_dict())

Once you have one activity, you can iterate through its exchanges. Brightway lets you inspect `production()`, `technosphere()`, and `biosphere()` separately.

In [ ]:
def preview_exchanges(exchanges, n=5):
    rows = []
    for exc in list(exchanges)[:n]:
        rows.append(
            {
                'input': exc.input['name'],
                'amount': exc['amount'],
                'unit': exc.input.get('unit'),
                'location': exc.input.get('location'),
                'categories': '::'.join(exc.input.get('categories') or ()),
                'type': exc['type'],
            }
        )
    return pd.DataFrame(rows)

print('Production exchanges')
display(preview_exchanges(selected_activity.production(), n=3))

print('Technosphere exchanges')
display(preview_exchanges(selected_activity.technosphere(), n=5))

print('Biosphere exchanges')
display(preview_exchanges(selected_activity.biosphere(), n=8))

You can use the same idea for any activity attributes (classifications, authors, year of publication), provided those are filled in. 

## Checkpoint 1

Pick one activity from the imported database, store it in `selected_activity`, and inspect a few production, technosphere, and biosphere exchanges.

Suggestion: start with a gasoline passenger car activity and refine by `location` if you get too many hits.

In [ ]:
# TODO
# Example pattern:
# hits = [
#     act for act in db
#     if 'passenger car' in act['name'].lower()
#     and any(term in act['name'].lower() for term in ['gasoline', 'petrol'])
# ]
# selected_activity = hits[0]
# preview_exchanges(selected_activity.technosphere(), n=3)
# preview_exchanges(selected_activity.biosphere(), n=3)

In [ ]:
if database_name not in bd.databases:
    print('Run the import cell first.')
else:
    db = bd.Database(database_name)
    hits = [
        act
        for act in db
        if 'passenger car' in act['name'].lower()
        and any(term in act['name'].lower() for term in ['gasoline', 'petrol'])
        and 'operation' in act['name'].lower()
    ]

    selected_activity = hits[0] if hits else db.random()

    print('Selected activity:', selected_activity['name'])
    print('Location:', selected_activity.get('location'))
    print('Unit:', selected_activity.get('unit'))

    print('First production exchanges:')
    display(preview_exchanges(selected_activity.production(), n=3))

    print('First technosphere exchanges:')
    display(preview_exchanges(selected_activity.technosphere(), n=3))

    print('First biosphere exchanges:')
    display(preview_exchanges(selected_activity.biosphere(), n=5))

## Checkpoint 2

Choose one of the LCIA methods already available in the installed project, ideally the climate-change indicator, and store it in `selected_method`.


In [ ]:
# TODO
# Example pattern:
# [m for m in bd.methods if 'climate change' in ' | '.join(m).lower()][:5]
# selected_method = ...


In [ ]:
preferred_method = ('EF v3.1', 'climate change', 'global warming potential (GWP100)')

if preferred_method in bd.methods:
    selected_method = preferred_method
else:
    climate_methods = [m for m in bd.methods if 'climate change' in ' | '.join(m).lower()]
    selected_method = climate_methods[0] if climate_methods else next(iter(bd.methods))

print('Selected method:', selected_method)
print('Method unit:', bd.Method(selected_method).metadata.get('unit'))


## 7) Run your first LCA and inspect the matrices

We now combine the selected activity and method into a complete LCA calculation, and then look at the matrix objects that Brightway builds behind the scenes.

In [ ]:
method = ('EF v3.1', 'climate change', 'global warming potential (GWP100)')

# bw2calc.LCA is the LCA calculation engine of `bw2calc`
# it expect a functional unit demand, and an LCIA method (optional)
lca = bc.LCA({selected_activity: 1}, selected_method)

# here, we solve the system (As=f) and obtain the inventory
lca.lci()

# and we multiply the inventory by the characterization matrix
lca.lcia()

print('Activity:', selected_activity['name'])
print('Method:', selected_method)
print('LCA score:', lca.score)


### 7.1 From the demand dict to the demand array

The argument passed to `bc.LCA(...)` is a Python dictionary: it is easy to read, but not yet in matrix form.  
After `lca.lci()`, Brightway has translated this demand into a numerical vector in product space: `lca.demand_array`.

In [ ]:
print('Demand dict:')
print(lca.demand)
print()
print('Demand array shape:', lca.demand_array.shape)
print('Number of non-zero entries in demand_array:', np.count_nonzero(lca.demand_array))

In [ ]:
demand_id = list(lca.demand.keys())[0]
demand_id

In [ ]:
# where is that demand in the demand vector?
demand_index = int(np.flatnonzero(lca.demand_array)[0])
demand_index

In [ ]:
# we have some reversed dictionaries that can 
# help us map indices (meaning position in a vector or matrix) to IDs
lca.dicts.product.reversed

In [ ]:
# let's check: is the position of our demand in the demand vector 
# leads to the ID we fetched from `lca.demand`?
lca.dicts.product.reversed[demand_index] == demand_id
# Cool!

In [ ]:
# hence, we now know who the functional unit is...
demanded_product = bd.get_activity(demand_id)
pd.Series(
    {
        'demanded product index': demand_index,
        'demanded product': demanded_product['name'],
        'location': demanded_product.get('location'),
        'amount in demand_array': lca.demand_array[demand_index],
    }
)

Mapping dicitonaries and their reversed forms are available for **products**, **activities** and **biosphere flows**.  
They allow us to know who is behind a specific position in a specific vector or matrix.

In [ ]:
list(lca.dicts.product.reversed.items())[:5]

In [ ]:
list(lca.dicts.activity.reversed.items())[:5]

In [ ]:
list(lca.dicts.biosphere.reversed.items())[:5]

### 7.2 Dictionaries that map matrix indices back to Brightway objects

The matrices only know row and column numbers. To go back from those numbers to Brightway nodes, use the dictionaries stored on the LCA object.

As seen above, those giving `id` -> `index` live in `lca.dicts.activity`, `lca.dicts.product`, and `lca.dicts.biosphere`.  
And those giving `index` -> `id` live in `lca.dicts.activity.reversed`, `lca.dicts.product.reversed`, and `lca.dicts.biosphere.reversed`

In [ ]:
selected_activity_index = lca.dicts.activity[selected_activity.id]

pd.DataFrame(
    [
        {
            'dictionary': 'activity',
            'size': len(lca.dicts.activity),
            'example matrix index': selected_activity_index,
            'example node name': bd.get_node(id=lca.dicts.activity.reversed[selected_activity_index])['name'],
        },
        {
            'dictionary': 'product',
            'size': len(lca.dicts.product),
            'example matrix index': demand_index,
            'example node name': bd.get_node(id=lca.dicts.product.reversed[demand_index])['name'],
        },
        {
            'dictionary': 'biosphere',
            'size': len(lca.dicts.biosphere),
            'example matrix index': 0,
            'example node name': bd.get_node(id=lca.dicts.biosphere.reversed[0])['name'],
        },
    ]
)

### 7.3 The technosphere matrix and the biosphere matrix

After `lca.lci()`, Brightway has built:

- `lca.technosphere_matrix`: the technosphere matrix $A$
- `lca.biosphere_matrix`: the biosphere matrix $B$

For a demanded product, a column of the technosphere matrix shows the products consumed and produced by that unit process. A column of the biosphere matrix shows the direct elementary flows of the corresponding activity.

In [ ]:
pd.DataFrame(
    [
        {'object': 'demand_array', 'shape': lca.demand_array.shape},
        {'object': 'technosphere_matrix', 'shape': lca.technosphere_matrix.shape},
        {'object': 'biosphere_matrix', 'shape': lca.biosphere_matrix.shape},
        {'object': 'supply_array', 'shape': lca.supply_array.shape},
        {'object': 'inventory', 'shape': lca.inventory.shape},
        {'object': 'characterization_matrix', 'shape': lca.characterization_matrix.shape},
        {'object': 'characterized_inventory', 'shape': lca.characterized_inventory.shape},
    ]
)

In [ ]:
# Let's slice the technosphere matrix to fetch all the rows (products) 
# at the column corresponding to our gasoline car transport activity.
technosphere_column = lca.technosphere_matrix[:, demand_index].tocoo()
technosphere_column

In [ ]:
# let's map those indices to product names and locations
technosphere_view = pd.DataFrame(
    [
        {
            'row index': row_idx,
            'col index': demand_index,
            'product': bd.get_node(id=rev_prod[row_idx])['name'],
            'location': bd.get_node(id=rev_prod[row_idx]).get('location'),
            'amount in A': amount,
        }
        for row_idx, amount in zip(technosphere_column.row, technosphere_column.data)
    ]
)

technosphere_view.sort_values('amount in A', ascending=False)

# notice how inputs are stored as negative entries in the technosphere/A matrix
# except the outgoing/production exchange, which is positive and on the diagonal (where row index == col index)

In [ ]:
# here too, let's slice the biosphere matrix to fetch all the rows (elementary flows) 
# at the column corresponding to our gasoline car transport activity.
biosphere_column = lca.biosphere_matrix[:, selected_activity_index].tocoo()

biosphere_view = pd.DataFrame(
    [
        {
            'row index': row_idx,
            'col index': demand_index,
            'biosphere flow': bd.get_node(id=rev_bio[row_idx])['name'],
            'categories': '::'.join(bd.get_node(id=rev_bio[row_idx]).get('categories') or ()),
            'amount in B': amount,
        }
        for row_idx, amount in zip(biosphere_column.row, biosphere_column.data)
    ]
)

biosphere_view.sort_values('amount in B', ascending=False).head(12)

For this notebook, two practical readings are enough:

- in the technosphere column, the positive diagonal entry is the reference product produced by the activity
- the other entries are the technosphere inputs used by that activity
- in the biosphere column, each row is a direct emission or resource use of that activity before upstream scaling

### 7.4 Supply array and inventory

`lca.supply_array` indicates how much each activity must run to meet the final demand $$\mathbf{A}\mathbf{s} = \mathbf{f}$$  

In [ ]:
# let's print the activities and the amount they must supply to satisfy the functional unit.
nonzero_supply = np.flatnonzero(lca.supply_array)

print(f"Number of supplying activities throughout the system: {nonzero_supply.shape}")

supply_view = pd.DataFrame(
    [
        {
            'activity': bd.get_node(id=lca.dicts.activity.reversed[idx])['name'],
            'location': bd.get_node(id=lca.dicts.activity.reversed[idx]).get('location'),
            'amount in supply_array': lca.supply_array[idx],
        }
        for idx in nonzero_supply
    ]
)

# let's print only the first 12 ones
supply_view.sort_values('amount in supply_array', ascending=False).head(12)

# These are 12 out of many activities along the supply chain that must provide an output to satisfy the functional unit

The inventory matrix then scales `lca.biosphere_matrix` by `lca.supply_array` $$\mathbf{G} = \mathbf{B}\,\mathrm{diag}(\mathbf{s})$$  
So summing a row in `lca.biosphere_matrix` gives a cradle-to-gate total for one pollutant.

In [ ]:
noxs = [
    flow
    for flow in bd.Database('ecoinvent-3.10-biosphere')
    if flow['name'] == 'Nitrogen oxides'
]
noxs

In [ ]:
[nox.id for nox in noxs]

But, let' be careful here: `lca.biosphere_matrix` isn't as large as the `biosphere` database, because `matrix_utils` (a sub-library) only fill in `lca.biosphere_matrix` with flows that are actuall used.

In [ ]:
len(bd.Database('ecoinvent-3.10-biosphere'))

In [ ]:
lca.biosphere_matrix.shape

In [ ]:
nox_rows = [lca.dicts.biosphere[nox.id] for nox in noxs if nox.id in lca.dicts.biosphere]
nox_rows

In [ ]:
print("NOx flow id", " | ", "NOx index in lca.biosphere_matrix")
for flow, flow_idx in zip(noxs, nox_rows):
    print(flow.id, " | ", flow_idx)

In [ ]:
emissions = []
for row in nox_rows:
    flow = bd.get_activity(lca.dicts.biosphere.reversed[row])
    emissions.append(
        [
            flow["name"],
            flow["categories"],
            flow["unit"],
            lca.inventory[row, selected_activity_index]
        ]
    )

print(f"Cradle-to-gate NOx emissions for activity with index {selected_activity_index}.")
pd.DataFrame(emissions, columns=["name", "categories", "unit", "amount"])

### 7.5 Characterized inventory

After `lca.lcia()`, Brightway applies the chosen characterization factors. The matrix `lca.characterized_inventory` has the same shape as `lca.inventory`, but its values are now in impact-score units rather than physical flow units. Summing all its cells gives the LCIA score.

In [ ]:
print('LCA score:', lca.score)

In [ ]:
print('LCA score:', np.sum(lca.characterization_matrix @ lca.inventory))

Let's sum `lca.characterized_inventory` column-wise (we collapsed activities)

In [ ]:
characterized_by_flow = np.asarray(lca.characterized_inventory.sum(axis=1)).ravel()

In [ ]:
# let's sort them by individual score and remove those that are zero and keep teh ten largest contributors
top_flow_indices = [
    idx for idx 
    in np.argsort(characterized_by_flow) 
    if characterized_by_flow[idx] != 0
]

# let's revert it, because np.argsort sorts ascendingly
top_flow_indices = top_flow_indices[::-1]
# and let's pick the ten largest
top_flow_indices = top_flow_indices[:10]

pd.DataFrame(
    [
        {
            'biosphere flow': bd.get_activity(id=lca.dicts.biosphere.reversed[idx])['name'],
            'categories': '::'.join(bd.get_activity(id=lca.dicts.biosphere.reversed[idx]).get('categories') or ()),
            'characterized contribution': characterized_by_flow[idx],
        }
        for idx in top_flow_indices
    ]
)

We can also sum row-wise to show the contribution of activities.

In [ ]:
characterized_by_activity = np.asarray(lca.characterized_inventory.sum(axis=0)).ravel()

In [ ]:
# let's sort them by individual score and remove those that are zero and keep teh ten largest contributors
top_flow_indices = [
    idx for idx 
    in np.argsort(characterized_by_activity) 
    if characterized_by_activity[idx] != 0
]

# let's revert it, because np.argsort sorts ascendingly
top_flow_indices = top_flow_indices[::-1]
# and let's pick the ten largest
top_flow_indices = top_flow_indices[:10]

pd.DataFrame(
    [
        {
            'biosphere flow': bd.get_activity(id=lca.dicts.activity.reversed[idx])['name'],
            'categories': '::'.join(bd.get_activity(id=lca.dicts.activity.reversed[idx]).get('categories') or ()),
            'characterized contribution': characterized_by_activity[idx],
        }
        for idx in top_flow_indices
    ]
)

## 8) Dedicated `ecoinvent` import overview

In this notebook, we only use the prepared `ecoinvent-3.10-biosphere` project archive as a shortcut for `biosphere3` and the LCIA methods.  
We do not import the `ecoinvent` technosphere database itself.

Key points:
- `ecoinvent` is not distributed in this repository.
- The full `ecoinvent 3.12 cutoff` technosphere import is run on Day 2, not in this notebook.
- The overall logic is the same: create an importer, apply strategies, inspect statistics, then write the database.
- In practice, `ecoinvent` uses dedicated `bw2io` importers.


In [ ]:
# Example preview. The actual import happens on Day 2.

# bi.import_ecoinvent_release(
#     version="3.12", 
#     system_model="cutoff", # other options are "consequential", "apos" and "EN15804"
#     username="xxxx",
#     password="xxxx",
#     biosphere_name="biosphere" # optional, otherwise a name is chosen for you
# )

## 9) Brief troubleshooting notes

- Project install fails: the first run of `bi.remote.install_project(...)` needs an internet connection to download the archive from `https://files.brightway.dev/`.
- Project already exists: the setup cell switches to the existing local project instead of downloading it again.
- Incomplete project contents: if the project exists but is missing `biosphere3` or the LCIA methods, delete that project or choose a fresh project name and rerun section 1.
- Workbook not found: confirm that `data/lci-bafu.xlsx` is present in the repository.
- `bafu` missing: rerun the BAFU import cells so that the workbook is parsed and written to the project.


## Recap

After this notebook, you should now know how to:

- bootstrap a `brightway` project from a prepared archive
- import a database with `ExcelImporter`
- inspect the LCIA methods already available in a project
- import your own LCIA method
- interpret the role of strategies, matching, and statistics
- search a database for activities and inspect exchanges
- run a first LCA from demand to score
- read `demand_array`, `technosphere_matrix`, `biosphere_matrix`, `supply_array`, and `inventory`
- use the LCA dictionaries to move between matrix indices and Brightway objects
- extract cradle-to-gate pollutant totals and interpret `characterized_inventory`
- recognize how the same import logic would extend to `ecoinvent`